# Data Collection — NIFTY 50 Pairs Trading

## Objective
Collect and clean historical price data for NIFTY 50 stocks in order to prepare the dataset for statistical arbitrage analysis such as:

- Correlation analysis
- Cointegration testing
- Spread construction
- Mean reversion trading

## Data Source
Yahoo Finance API via `yfinance`.

## Frequency
Daily adjusted close prices.

## Time Period
2015 – Present


In [1]:
# Import required libraries

import yfinance as yf
import pandas as pd
import numpy as np




## NIFTY 50 Constituents

We manually define the list of NIFTY 50 tickers available on Yahoo Finance.


In [2]:
nifty_50 = [
"ADANIENT.NS","ADANIPORTS.NS","APOLLOHOSP.NS","ASIANPAINT.NS",
"AXISBANK.NS","BAJAJ-AUTO.NS","BAJFINANCE.NS","BAJAJFINSV.NS",
"BEL.NS","BAJAJHLDNG.NS","BHARTIARTL.NS","BPCL.NS","BRITANNIA.NS",
"CIPLA.NS","COALINDIA.NS","DRREDDY.NS","EICHERMOT.NS",
"GRASIM.NS","HCLTECH.NS","HDFCBANK.NS","HDFCLIFE.NS",
"HEROMOTOCO.NS","HINDALCO.NS","HINDUNILVR.NS","ICICIBANK.NS",
"ITC.NS","INDUSINDBK.NS","INFY.NS","JSWSTEEL.NS",
"KOTAKBANK.NS","LT.NS","M&M.NS","MARUTI.NS",
"NESTLEIND.NS","NTPC.NS","ONGC.NS","POWERGRID.NS",
"RELIANCE.NS","SBILIFE.NS","SBIN.NS","SUNPHARMA.NS",
"TCS.NS","TATACONSUM.NS","TATAMOTORS.NS","TATASTEEL.NS",
"TECHM.NS","TITAN.NS","ULTRACEMCO.NS","UPL.NS",
"WIPRO.NS"
]

In [3]:
# Define data download period

start_date = "2015-01-01"
end_date = None   # None means latest available data


In [4]:
# Download historical price data for all NIFTY 50 stocks

raw_data = yf.download(
    tickers=nifty_50,
    start=start_date,
    end=end_date,
    progress=False,
    group_by="ticker"
)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


## Data Cleaning

Steps performed:

1. Extract adjusted close prices  
2. Handle missing columns  
3. Remove invalid tickers  
4. Drop missing rows  
5. Sort index by date


In [5]:
# Extract adjusted close prices

close_prices = pd.DataFrame()

for ticker in nifty_50:
    
    try:
        if "Adj Close" in raw_data[ticker].columns:
            close_prices[ticker] = raw_data[ticker]["Adj Close"]

        elif "Close" in raw_data[ticker].columns:
            close_prices[ticker] = raw_data[ticker]["Close"]

    except Exception as e:
        print(f"Skipping {ticker}: {e}")


In [6]:
# Remove empty columns (stocks with no data)
close_prices.dropna(axis=1, how="all", inplace=True)

# Remove rows containing missing values
close_prices.dropna(inplace=True)


In [7]:
# Ensure data is ordered by date
close_prices = close_prices.sort_index()


In [8]:
# Dataset validation

print(f"Number of stocks: {close_prices.shape[1]}")
print(f"Number of observations: {close_prices.shape[0]}")
print(f"Date range: {close_prices.index.min()} to {close_prices.index.max()}")


Number of stocks: 49
Number of observations: 2061
Date range: 2017-11-17 00:00:00 to 2026-03-20 00:00:00


### Missing Data Check

Some tickers may fail to download due to temporary Yahoo Finance API issues
or incomplete historical records. Such tickers are automatically removed
to ensure the dataset used for analysis is clean and consistent.


In [9]:
missing_tickers = set(nifty_50) - set(close_prices.columns)

print("Missing tickers:", missing_tickers)
print("Final number of stocks:", len(close_prices.columns))


Missing tickers: {'TATAMOTORS.NS'}
Final number of stocks: 49


In [10]:
# Save cleaned dataset

close_prices.to_csv("../data/processed/nifty50_close_prices.csv",index_label="Date")

print("Dataset saved successfully")


Dataset saved successfully


## Output

The cleaned dataset is saved to:

data/processed/nifty50_close_prices.csv

This dataset will be used in the next notebook for correlation analysis.
